# 41장 · Streamlit 세션과 LangGraph 체크포인트

화면에 보이는 현재 답변과 그래프에 쌓이는 대화 기록은 수명이 다릅니다. 어떤 사용자와 스레드의 값을 갱신하는지 확인합니다.

Streamlit 대화 이력과 LangGraph의 thread_id를 연결합니다. 지난 질문은 대화에 보존하고, 현재 답변의 출처 패널은 이번 검색 결과로 교체합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 화면 상태와 그래프 저장 상태

LangGraph에 InMemorySaver를 연결해 스레드별로 질문을 기록합니다. 화면 담당 코드는 현재 대화의 thread_id를 전달합니다. 셀 01~03은 도서관 고정 자료를 검색합니다. 셀 04~05에서는 학생이 입력한 안내문 한 건을 같은 그래프 호출에 전달합니다.


### 셀 01 · 체크포인터가 있는 그래프 구성

history에는 누적 규칙을 붙이고 sources에는 현재 결과만 덮어쓰도록 합니다.


In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class Session(TypedDict):
    question: str
    sources: list[dict]
    history: Annotated[list[str], add]

def process(state):
    return {'sources': retrieve(state['question'], 1), 'history': [state['question']]}

builder = StateGraph(Session)
builder.add_node('process', process)
builder.add_edge(START, 'process')
builder.add_edge('process', END)
engine = builder.compile(checkpointer=InMemorySaver())

def send(thread, question):
    return engine.invoke({'question': question}, {'configurable': {'thread_id': thread}})

first = send('A', '대출 기간은?')
print(first['sources'][0]['id'])

library-1


A의 현재 결과는 library-1입니다. 반복 실행으로 새 엔진을 만들면 메모리 저장소도 다시 만들어집니다.


### process 하나로 구성한 그래프

<p>START 다음에 실행되는 노드는 process 하나입니다. retrieve는 이 노드 안에서 부르는 함수입니다. process가 현재 질문의 sources와 한 항목짜리 history를 반환하면 그래프가 각 필드의 규칙에 맞춰 갱신하고 END에 도착합니다.</p>


### 셀 02 · 현재 대화 출처 초기화

A가 주차 질문을 보낸 직후의 기록과 출처를 읽으세요.


In [2]:
second = send('A', '주차 요금은?')
print(second['history'])
print(second['sources'])

['대출 기간은?', '주차 요금은?']
[]


질문 두 개가 기록에 있고 현재 출처는 비었습니다. 프런트엔드가 첫 답의 출처를 계속 들고 있다면 화면만 잘못된 상태일 수 있습니다.

질문 이력에는 대출 기간과 주차 요금이 둘 다 남았는데 현재 출처 목록은 비었습니다. 화면이 되살려야 할 두 상태의 수명이 다르기 때문입니다. 대화 목록은 이어 가되, 출처 패널은 방금 처리한 질문의 결과로 교체하세요.


## 사용자와 thread_id로 대화 분리

화면의 새 대화 버튼은 새 ID와 빈 현재 화면을 준비합니다. 다른 사용자의 thread_id를 재사용하면 이전 상태를 이어 읽으므로 이 노트북의 A와 B 같은 예시 ID를 운영 인증에 쓰면 안 됩니다.


### 과제 03 · thread_id별 기록 분리

B라는 새 대화에 휴관일을 묻고 B의 질문 목록과 현재 쪽수를 표시하세요.


In [3]:
# 다른 ID로 send를 호출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>other = send(&#x27;B&#x27;, &#x27;휴관일은?&#x27;)
print(other[&#x27;history&#x27;], other[&#x27;sources&#x27;][0][&#x27;page&#x27;])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;휴관일은?&#x27;] 2</code></pre>
<p>B에는 휴관 질문 한 개와 2쪽만 보입니다. A의 두 질문이 나타났다면 호출한 ID와 화면에 남긴 기록을 각각 살펴보세요.</p>
</details>


## Streamlit 폼에서 그래프 호출

제공 앱은 도서관 자료에 더해 폼에 입력한 안내문 한 건을 검색할 수 있습니다. 질문에 지정한 검색어가 들어 있으면 입력한 안내문을 먼저 보여 줍니다. 앞서 본 처리 중 UI에서 결정하는 것은 대화 ID와 현재 표시할 결과입니다. 새 대화 버튼은 새 ID를 만듭니다. 수업용 예제에는 로그인·운영 저장소가 없습니다.


### 셀 04 · 폼 상태와 대화 ID 연결

st.session_state.thread를 설정하는 줄과 engine.invoke에 같은 ID를 건네는 줄을 찾아보세요.

새 대화 버튼은 새 ID를 만들고 current를 비웁니다. 그래프 엔진을 준비하는 코드는 prepare_engine에서, 폼 제출 결과를 현재 화면에 넘기는 코드는 if submit 안에서 확인할 수 있습니다.


In [4]:
ui_source = """records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from typing import TypedDict
from uuid import uuid4
import streamlit as st
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated
from operator import add

class Chat(TypedDict):
    question: str
    note: dict | None
    sources: list[dict]
    history: Annotated[list[str], add]

def process(state):
    question = state["question"]
    note = state["note"]
    if note and note["keyword"] in question:
        sources = [note]
    else:
        sources = retrieve(question, 1)
    return {"sources": sources, "history": [question]}

@st.cache_resource
def prepare_engine():
    builder = StateGraph(Chat)
    builder.add_node("process", process)
    builder.add_edge(START, "process")
    builder.add_edge("process", END)
    return builder.compile(checkpointer=InMemorySaver())

new_chat = st.button("새 대화")
if "thread" not in st.session_state or new_chat:
    st.session_state.thread = str(uuid4())
    st.session_state.current = None
engine = prepare_engine()
with st.form("ask"):
    question = st.text_input("질문")
    keyword = st.text_input("추가 문서 검색어 (선택)").strip()
    notice = st.text_area("추가 안내문 (선택)").strip()
    submit = st.form_submit_button("질문 보내기")
if submit and question.strip():
    config = {"configurable": {"thread_id": st.session_state.thread}}
    if bool(keyword) != bool(notice) or (keyword and keyword not in notice):
        st.session_state.current = None
        st.warning("추가 안내문과 그 안에 들어 있는 검색어를 함께 입력하세요.")
    else:
        note = {"id": "session-note", "keyword": keyword, "text": notice} if notice else None
        with st.spinner("안내문 검색 중"):
            st.session_state.current = engine.invoke({"question": question, "note": note}, config)
if st.session_state.current:
    state = st.session_state.current
    st.text("지난 질문: " + " / ".join(state["history"]))
    if state["sources"]:
        current = state["sources"][0]
        st.write(current["text"])
        location = f"{current['page']}쪽" if "page" in current else "직접 입력한 안내문"
        st.caption(f"근거 [{current['id']}] {location}")
    else:
        st.info("제공된 자료에서 확인할 수 없습니다.")
"""
import ast
ast.parse(ui_source)
print("앱 구문 확인")

앱 구문 확인


구문 검사는 ui_source가 파이썬 코드로 해석되는지 확인합니다. 다음 셀에서 AppTest에 같은 문자열을 전달해 연속 제출 전후의 대화 기록과 출처를 대조합니다.


### 화면과 그래프가 보관하는 값

<p>prepare_engine은 @st.cache_resource를 사용해 그래프와 InMemorySaver를 재사용합니다. Streamlit의 session_state에는 현재 thread와 마지막 실행 결과 current가 들어갑니다. 새 대화는 새 ID를 할당하고 current를 비웁니다. 과거 스레드의 체크포인트는 같은 저장소에 남습니다. 메모리 저장소를 새로 만들거나 프로세스를 종료하면 이 예제의 체크포인트는 사라집니다.</p>


### 폼 제출값을 그래프 입력으로 보내기

<p>추가 문서를 사용하려면 본문과 그 안에 있는 검색어를 함께 입력합니다. 검색어만 남기고 본문을 지우면 경고를 표시하고 이전 출처를 지웁니다. 둘 다 비워 두면 기존 자료만 검색합니다. 직접 입력한 안내문에는 쪽 번호가 없으므로 화면에도 쪽 번호를 만들지 않습니다.</p>


### 셀 05 · 연속 제출 후 화면 상태

AppTest에서 먼저 고정 자료로 대출과 주차를 묻습니다. 이어 “첫 30분 주차는 무료입니다.”를 등록하고 같은 주차 질문을 다시 제출하세요. 이전에 비었던 출처가 어떻게 바뀌는지 확인합니다.


In [5]:
from importlib.util import find_spec
if find_spec("streamlit") is None:
    print("Streamlit이 설치된 수업 커널에서 화면 연결을 확인하세요.")
else:
    from streamlit.testing.v1 import AppTest
    ui = AppTest.from_string(ui_source).run()
    ui.text_input[0].set_value("대출 기간은?")
    ui.button[1].click().run()
    assert not ui.exception and ui.caption[0].value.endswith("1쪽")
    ui.text_input[0].set_value("주차 요금은?")
    ui.button[1].click().run()
    assert not ui.exception and not ui.caption
    print(ui.text[0].value, ui.info[0].value)
    ui.text_input[1].set_value("주차")
    ui.text_area[0].set_value("첫 30분 주차는 무료입니다.")
    ui.text_input[0].set_value("주차 요금은?")
    ui.button[1].click().run()
    assert not ui.exception and ui.caption[0].value.endswith("직접 입력한 안내문")
    print(ui.markdown[0].value, ui.caption[0].value)
    ui.text_area[0].set_value("")
    ui.button[1].click().run()
    assert not ui.exception and ui.warning and not ui.caption

Streamlit이 설치된 수업 커널에서 화면 연결을 확인하세요.


고정 자료만 쓸 때는 두 번째 질문에 출처가 없습니다. 안내문을 추가한 뒤에는 같은 질문에 “첫 30분 주차는 무료입니다.”와 직접 입력한 출처가 함께 나옵니다. 마지막 제출에서 안내문만 지우면 경고가 뜨고 출처는 사라집니다. 설치 안내만 출력되면 위젯 검사는 실행되지 않은 상태입니다.


### 과제 06 · 다른 대화를 시작한 뒤 원래 스레드 다시 열기

서로 다른 두 스레드에 한 건씩 질문한 뒤 첫 스레드에 새 질문을 보내세요. 누적된 질문과 현재 출처를 출력하세요. 재실행해도 앞선 실험 기록이 섞이지 않게 이번 실험 전용 ID를 만드세요.


In [ ]:
from uuid import uuid4
first_thread, second_thread = str(uuid4()), str(uuid4())
# 첫 스레드에는 대출과 주차, 둘째에는 휴관을 보내세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>from uuid import uuid4
first_thread, second_thread = str(uuid4()), str(uuid4())
send(first_thread, &#x27;대출 기간은?&#x27;)
send(second_thread, &#x27;휴관일은?&#x27;)
current = send(first_thread, &#x27;주차 요금은?&#x27;)
print(current[&#x27;history&#x27;])
print(current[&#x27;sources&#x27;])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;대출 기간은?&#x27;, &#x27;주차 요금은?&#x27;]
[]</code></pre>
<p>첫 스레드의 두 질문이 원래 순서로 남고 다른 대화에서 물은 휴관일은 들어오지 않았습니다. 최신 질문에는 근거가 없어 sources가 비었습니다. 같은 그래프 객체를 재사용하면서 저장할 대화만 ID로 골랐습니다.</p>
</details>


## 돌아보기

문서 연결이 실패하거나 검색 오류가 발생한 경우에는 현재 답과 출처를 먼저 지우세요. 사용자가 지난 질문을 다시 읽을 수 있도록 과거 기록은 분리해 둡니다.
